In [1]:
from datetime import datetime
import xml.etree.ElementTree as ET

from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, 
    FloatType, DateType)

# columns for the DataFrame
COL_NAMES = ['book_id', 'author', 'title', 'genre', 'price', 'publish_date',
            'description']
ELEMENTS_TO_EXTRAT = [c for c in COL_NAMES if c != 'book_id']

def set_schema():
    """
    Define the schema for the DataFrame
    """
    schema_list = []
    for c in COL_NAMES:
        if c == 'price':
            schema_list.append(StructField(c, FloatType(), True))
        elif c == 'publish_date':
            schema_list.append(StructField(c, DateType(), True))
        else:
            schema_list.append(StructField(c, StringType(), True))
    
    return StructType(schema_list)

def parse_xml(rdd):
    """
    Read the xml string from rdd, parse and extract the elements,
    then return a list of list.
    """
    results = []
    root = ET.fromstring(rdd[0])

    for b in root.findall('book'):
        rec = []
        rec.append(b.attrib['id'])
        for e in ELEMENTS_TO_EXTRAT:
            if b.find(e) is None:
                rec.append(None)
                continue
            value = b.find(e).text
            if e == 'price':
                value = float(value)
            elif e == 'publish_date':
                value = datetime.strptime(value, '%Y-%m-%d')
            rec.append(value)
        results.append(rec)

    return results

In [2]:
# define the schema
my_schema = set_schema()
# read each xml file as one row, then convert to RDD
file_rdd = spark.read.text("gs://lei-data/books.xml", wholetext=True).rdd
# parse xml tree, extract the records and transform to new RDD
records_rdd = file_rdd.flatMap(parse_xml)
# convert RDDs to DataFrame with the pre-defined schema
book_df = records_rdd.toDF(my_schema)

book_df.printSchema()

root
 |-- book_id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- price: float (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- description: string (nullable = true)



In [3]:
book_df.show()

+-------+-------------------+------------------+-----+-----+------------+--------------------+
|book_id|             author|             title|genre|price|publish_date|         description|
+-------+-------------------+------------------+-----+-----+------------+--------------------+
|  bk101|         Loro, Lars|Doing good is good| null| null|        null|This is the place...|
|  bk102|Poderevyansky, Alex|Misterious Ambalus| null| null|        null| Misterious Ambalus.|
|  bk119|        Feng, Jason|        Playground| null| null|        null|This is the place...|
+-------+-------------------+------------------+-----+-----+------------+--------------------+



In [4]:
# Saving the data to BigQuery
# 'com.google.cloud.spark.bigquery'
table = "lorelee-keep-312708.lei_staging.books"
book_df.write.format('bigquery') \
  .option("temporaryGcsBucket","temp-dataproc-storage") \
  .option('table', table) \
  .mode("overwrite") \
  .save()

print("Finished the workflow")

Finished the workflow


In [5]:
# write to csv
spark_write_method = "overwrite"
output_path="gs://lei-data/output.csv"
book_df.write.mode(spark_write_method).csv(output_path, compression='gzip')

print("Saved the output as a gzipped csv file")

Saved the output as a gzipped csv file
